# Agent comparison

Scores each `agent_name` against a fixed ground truth and compares them
head to head.

**This notebook inverts the assumption the previous one made.** There,
`agent_name` was treated as repeat runs of one cell. Here it is treated
as *different candidate patches*, which means:

- Ground truth (F2P / P2P) must come from a **reference/gold source
  only**, never pooled across agents. An agent cannot be graded against
  a rubric it helped write.
- Everything else is a candidate to be scored against that rubric.

Cell 3 checks which of those two worlds your data actually lives in.
Set `GOLD_AGENT` in cell 2 before running.

In [ ]:
from __future__ import annotations

import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Reuses the tested pipeline from the previous notebook. Keep
# test_split.py next to this file.
from test_split import Columns, classify_tests

pd.set_option('display.width', 120)

# --- Schema / config: edit to match your data ---

INSTANCE: str = 'instance_id'
PATCH: str = 'patch_type'
AGENT: str = 'agent_name'
TEST: str = 'test_name'
PASSED: str = 'passed'

PRE: str = 'before_patch'
POST: str = 'gold'

# The reference run that defines ground truth. Everything else is a
# candidate to be scored.
GOLD_AGENT: str = 'GOLD'

MODES: list[str] = [
    'RESOLVED',
    'F2P_FAIL',
    'P2P_REGRESSION',
    'BOTH',
]

## 1. Load

In [ ]:
# Replace with your own load.
df = pd.read_parquet('s3-sync/test-results/')

df[PASSED] = df[PASSED].astype(bool)

print(df.shape)
df.head()

## 2. Does the data support an agent comparison at all?

Three things have to hold. If any fails, the leaderboard below will be
meaningless rather than wrong-looking:

1. `GOLD_AGENT` exists and covers **both** `PRE` and `POST`. Without a
   pre-patch reference there is no F2P set.
2. Candidate agents have `POST` rows.
3. Candidate agents do **not** define their own pre-patch state.

In [ ]:
coverage = pd.crosstab(df[AGENT], df[PATCH])
print(coverage)
print()

assert GOLD_AGENT in coverage.index, f'no agent named {GOLD_AGENT!r}'

candidates = sorted(set(df[AGENT].unique()) - {GOLD_AGENT})
print('candidate agents:', candidates)

# If candidates carry their own pre-patch rows, decide deliberately
# whether to ignore them (the gold pre-patch state is the honest
# baseline) rather than letting them leak into ground truth.
cand_pre = coverage.loc[candidates, PRE].sum() if candidates else 0
print(f'candidate rows on the {PRE!r} side (ignored):', int(cand_pre))

## 3. Ground truth from the reference run only

`classify_tests` collapses repeat runs, quarantines flaky tests, and
splits on the pre -> post transition. Feeding it *only* gold rows is
what keeps the rubric independent of the agents being graded.

In [ ]:
df['agent_name'].unique()

In [ ]:
cols = Columns(
    instance=INSTANCE,
    patch_type=PATCH,
    test_name=TEST,
    passed=PASSED,
)

gold = df[(df[AGENT] == GOLD_AGENT) | (df[AGENT] == 'BASE')]
truth = classify_tests(gold, pre_label=PRE, post_label=POST, columns=cols)

f2p, p2p = truth.fail_to_pass, truth.pass_to_pass

print('instances with >=1 F2P:', len(f2p))
print('F2P tests total:', sum(len(v) for v in f2p.values()))
print('P2P tests total:', sum(len(v) for v in p2p.values()))
print('flaky (excluded):', sum(len(v) for v in truth.flaky.values()))

# Only instances with a non-empty F2P set are gradeable: nothing else
# can demonstrate the bug was fixed.
gradeable = sorted(f2p)
print('gradeable instances:', len(gradeable))

In [ ]:
# The rubric, in long form: every test each instance must satisfy.
req = pd.DataFrame(
    [(i, t, 'F2P') for i, ts in f2p.items() for t in ts]
    + [(i, t, 'P2P') for i, ts in p2p.items() for t in ts],
    columns=[INSTANCE, TEST, 'kind'],
)
req = req[req[INSTANCE].isin(gradeable)]

print(req['kind'].value_counts())
req.head()

## 4. Collapse each candidate's post-patch runs

Same strict rule as before: a test passes only if it passed in every
run of that agent.

In [ ]:
cand = df[(df[AGENT] != GOLD_AGENT) & (df[PATCH] == POST)]

verdict = (
    cand.groupby([INSTANCE, AGENT, TEST], observed=True)[PASSED]
    .all()
    .reset_index()
)

print(f'{len(cand)} rows -> {len(verdict)} (instance, agent, test) cells')
verdict.head()

### Attempted vs. missing

An agent-instance pair with no rows at all never ran — a crash, a
timeout, a harness error. That is **not** the same as failing, and
lumping the two together silently rewards agents that fail to launch.
Section 5 reports both, then scores on the common subset.

In [ ]:
attempted = set(
    map(tuple, verdict[[INSTANCE, AGENT]].drop_duplicates().to_numpy())
)

cov = pd.DataFrame(
    [[(i, a) in attempted for a in candidates] for i in gradeable],
    index=gradeable,
    columns=candidates,
)

print('gradeable instances:', len(cov))
print()
print('attempted per agent:')
print(cov.sum().sort_values(ascending=False))

common = cov.index[cov.all(axis=1)].tolist()
print()
print(f'attempted by every agent (common subset): {len(common)}')

## 5. Score each candidate against the rubric

In [ ]:
grid = req.merge(pd.DataFrame({AGENT: candidates}), how='cross')
scored = grid.merge(verdict, on=[INSTANCE, AGENT, TEST], how='left')

# A required test with no result did not pass. This is the conservative
# reading: a crashed run gets no credit.
scored['no_result'] = scored[PASSED].isna()
scored[PASSED] = scored[PASSED].fillna(False).astype(bool)

print('required (instance, agent, test) checks:', len(scored))
print('with no result recorded:', int(scored['no_result'].sum()))
scored.head()

In [ ]:
is_f2p = scored['kind'] == 'F2P'

scored['f2p_req'] = is_f2p
scored['f2p_pass'] = is_f2p & scored[PASSED]
scored['p2p_req'] = ~is_f2p
scored['p2p_pass'] = (~is_f2p) & scored[PASSED]

summary = (
    scored.groupby([INSTANCE, AGENT], observed=True)[
        ['f2p_req', 'f2p_pass', 'p2p_req', 'p2p_pass']
    ]
    .sum()
    .reset_index()
)

# Resolved == every F2P passes AND no P2P regressed. Both halves
# matter: an agent that deletes the test file passes F2P vacuously.
summary['f2p_ok'] = summary['f2p_pass'] == summary['f2p_req']
summary['p2p_ok'] = summary['p2p_pass'] == summary['p2p_req']
summary['resolved'] = summary['f2p_ok'] & summary['p2p_ok']

summary['mode'] = np.select(
    [
        summary['resolved'],
        ~summary['f2p_ok'] & ~summary['p2p_ok'],
        ~summary['f2p_ok'],
    ],
    ['RESOLVED', 'BOTH', 'F2P_FAIL'],
    default='P2P_REGRESSION',
)
summary['attempted'] = [
    (i, a) in attempted
    for i, a in zip(summary[INSTANCE], summary[AGENT], strict=True)
]

summary.head()

## 6. Leaderboard

Scored on the common subset only, so every agent faces the same
instances. The Wilson interval is there to stop small differences being
over-read: with a few hundred instances, gaps of a couple of points are
usually noise.

In [ ]:
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """Wilson score interval for a binomial proportion.

    Preferred over the normal approximation because resolve rates sit
    near 0 or 1, where the normal interval leaves the unit range.

    Args:
        k: Number of successes.
        n: Number of trials.
        z: Standard-normal quantile; 1.96 gives 95% coverage.

    Returns:
        The (lower, upper) bound.
    """
    if n == 0:
        return (float('nan'), float('nan'))
    p = k / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return ((centre - half) / denom, (centre + half) / denom)


sub = summary[summary[INSTANCE].isin(common)]

board = (
    sub.groupby(AGENT)['resolved'].agg(resolved='sum', n='size').reset_index()
)
board['rate'] = board['resolved'] / board['n']
board[['lo', 'hi']] = [
    wilson(k, n) for k, n in zip(board['resolved'], board['n'], strict=True)
]
board = board.sort_values('rate', ascending=False).reset_index(drop=True)

board

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
y = np.arange(len(board))
err = np.vstack(
    [
        board['rate'] - board['lo'],
        board['hi'] - board['rate'],
    ]
)
ax.barh(y, board['rate'], xerr=err, capsize=4)
ax.set_yticks(y, board[AGENT])
ax.invert_yaxis()
ax.set_xlabel('resolve rate')
ax.set_title(f'Resolve rate on {len(common)} common instances (95% CI)')
for i, r in enumerate(board['rate']):
    ax.text(r, i, f' {r:.1%}', va='center')
plt.tight_layout()

## 7. How they fail

Splitting failures apart is the useful part. An agent losing to
`P2P_REGRESSION` is over-editing and breaking working code; one losing
to `F2P_FAIL` simply did not fix the bug. Those call for opposite
remedies.

In [ ]:
modes = (
    pd.crosstab(sub[AGENT], sub['mode'], normalize='index')
    .reindex(columns=MODES, fill_value=0.0)
    .reindex(board[AGENT])
)

fig, ax = plt.subplots(figsize=(8, 4))
modes.plot.barh(stacked=True, ax=ax)
ax.invert_yaxis()
ax.set_xlabel('share of instances')
ax.set_ylabel('')
ax.set_title('Outcome breakdown per agent')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

(modes * 100).round(1)

In [ ]:
# Partial credit: how close did the misses get? A high F2P pass rate
# among unresolved instances means the agents are near, not lost.
near = sub[~sub['resolved']].copy()
near['f2p_frac'] = near['f2p_pass'] / near['f2p_req']

fig, ax = plt.subplots(figsize=(7, 4))
for agent, grp in near.groupby(AGENT):
    ax.hist(grp['f2p_frac'], bins=20, alpha=0.5, label=agent)
ax.set_xlabel('fraction of F2P tests passed (unresolved only)')
ax.set_ylabel('instances')
ax.set_title('How close the failures came')
ax.legend()
plt.tight_layout()

near.groupby(AGENT)['f2p_frac'].describe()[['mean', '50%', 'max']]

## 8. Instance difficulty

In [ ]:
mat = (
    sub.pivot(index=INSTANCE, columns=AGENT, values='resolved')
    .reindex(columns=board[AGENT])
    .astype(bool)
)

solved_by = mat.sum(axis=1)
print('instances by number of agents solving them:')
print(solved_by.value_counts().sort_index())
print()
print('solved by nobody:', int((solved_by == 0).sum()))
print('solved by everybody:', int((solved_by == len(mat.columns)).sum()))

# Instances nobody or everybody solves carry no discriminating signal;
# the contested middle is where the agents actually differ.
contested = solved_by[(solved_by > 0) & (solved_by < len(mat.columns))]
print('contested:', len(contested))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = solved_by.value_counts().sort_index()
ax.bar(counts.index, counts.values)
ax.set_xlabel('number of agents that resolved the instance')
ax.set_ylabel('instances')
ax.set_title('Instance difficulty')
ax.set_xticks(range(len(mat.columns) + 1))
plt.tight_layout()

## 9. Head to head

In [ ]:
# Unique solves: instances an agent got that no other agent did.
unique = {
    a: int((mat[a] & ~mat.drop(columns=a).any(axis=1)).sum())
    for a in mat.columns
}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(unique), list(unique.values()))
ax.set_ylabel('instances solved by this agent alone')
ax.set_title('Unique solves')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()

pd.Series(unique, name='unique_solves').sort_values(ascending=False)

In [ ]:
agree = pd.DataFrame(
    [
        [float((mat[a] == mat[b]).mean()) for b in mat.columns]
        for a in mat.columns
    ],
    index=mat.columns,
    columns=mat.columns,
)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(agree, vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(len(agree)), agree.columns, rotation=45, ha='right')
ax.set_yticks(range(len(agree)), agree.index)
for i in range(len(agree)):
    for j in range(len(agree)):
        ax.text(
            j, i, f'{agree.iat[i, j]:.2f}', ha='center', va='center', color='w'
        )
ax.set_title('Agreement (same verdict per instance)')
fig.colorbar(im, ax=ax)
plt.tight_layout()

agree.round(3)

### Is the gap real?

Comparing two agents on the *same* instances is a paired problem, so
the unpaired CIs in section 6 are the wrong test — they discard the
pairing and lose power. Exact McNemar uses only the instances where the
two disagree, which is the only place information about the difference
lives.

In [ ]:
def mcnemar_exact(b: int, c: int) -> float:
    """Two-sided exact McNemar p-value for paired binary outcomes.

    Only the discordant pairs carry information about the difference,
    so the test asks whether b successes out of b + c is consistent
    with a fair coin.

    Args:
        b: Instances the first agent resolved and the second did not.
        c: Instances the second resolved and the first did not.

    Returns:
        The two-sided p-value.
    """
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2**n
    return min(1.0, 2 * tail)


rows = []
for a, b in itertools.combinations(mat.columns, 2):
    only_a = int((mat[a] & ~mat[b]).sum())
    only_b = int((mat[b] & ~mat[a]).sum())
    rows.append(
        {
            'agent_a': a,
            'agent_b': b,
            'only_a': only_a,
            'only_b': only_b,
            'diff': (mat[a].mean() - mat[b].mean()),
            'p_value': mcnemar_exact(only_a, only_b),
        }
    )

pairs = pd.DataFrame(rows).sort_values('p_value')
pairs['significant_05'] = pairs['p_value'] < 0.05
pairs

## 10. Export

In [ ]:
board.to_csv('agent_leaderboard.csv', index=False)
summary.to_csv('agent_instance_summary.csv', index=False)

print('wrote agent_leaderboard.csv, agent_instance_summary.csv')
print()
print(
    f'scored {len(candidates)} agents on {len(common)} common '
    f'instances out of {len(gradeable)} gradeable'
)